# Towards Generative Modeling of CO₂ Geological Storage with Latent Conditional Diffusion Models

|| **Author: Vittoria De Pellegrini** || **Year: 2025** ||
**Institution: King Abdullah University of Science and Technology (KAUST)** ||

This notebook provides a comprehensive guide for validating a **Latent Conditional Diffusion Model** for Pressure BuildUp Dataset.

### ⚠️ Define Normalization Strategy

To normalize our dataset to the range $[-1, 1]$, we utilize **Local Normalization**. This approach involves normalizing each sample independently based on its own minimum and maximum values. We adopt local normalization because the range of values differs significantly between samples. The formula for local normalization is calculated as follows: 

$$x_{\text{norm}} = 2 \times \frac{x - \min(x)}{\max(x) - \min(x)} - 1\tag{1}$$

Where:
- $\min(x)$ and $\max(x)$ are the minimum and maximum values of the each sample $x$ in the dataset $X = \{x_1, x_2, \dots, x_N\}$, where $N$ is the number of samples.
- $x_{\text{norm}}$ denotes the normalized value, which will falls in the range $[-1, 1]$.

Note: 
- All 12 conditions are stored as 2D maps. If some conditions have fixed values, we do not apply any normalization. 

### 📚⬇ Import Libraries and Functions 

In [ ]:
# Import libraries 
import warnings
warnings.filterwarnings('ignore')

import os 
import torch
from torch.utils.data import DataLoader, TensorDataset

import sys
sys.path.append('/.../')

# Import functions 
from dmfco2 import(
    set_seed,     
    dnorm_dP,
    visualize_reverse_generation_dP_val,
    VQVAE,
    Linear_Variance_Scheduler,
    Cosine_Variance_Scheduler,
    Conditional_Unet
)

### ❗️ Identify Number of Devices (Multi-GPU Training with Accelerate)

In [ ]:
# Print the number of GPUs available
num_devices = torch.cuda.device_count()
print("Number of devices: ", num_devices)

# Loop through all available devices and print their names
for i in range(num_devices):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")
    
# Print current device
print("Current device: ", torch.cuda.current_device())

### ❗️Set Seed for Reproducible Results

In [ ]:
# Set seed
set_seed(seed=42)

### 📕 Define Inputs (Conditions) and Outputs (Targets) Variables for Pressure BuildUp Validation Dataset

In [ ]:
# Define validation dataset directory path
data_dir_dP_val = '/.../co2_dataset/validation_dataset_pressure_buildup'

In [ ]:
# Define inputs (conditions) and outputs (targets) variables
val_dP_inputs = torch.load(f'{data_dir_dP_val}/dP_val_inputs.pt')
val_dP_outputs = torch.load(f'{data_dir_dP_val}/dP_val_outputs.pt')

print(val_dP_inputs.shape) # Shape: [num_samples, height, width, time_steps, num_features]
print(val_dP_outputs.shape)

In [ ]:
# Inputs -> Extract last time frame 30.0 y (index -1) and we consider all conditions which are 12 
val_dP_inputs  = val_dP_inputs[:, :, :, -1, :]  

# Outputs -> Extract last time frame 30.0 y
val_dP_outputs = val_dP_outputs[:, :, :, -1] 

print("New shape:", val_dP_inputs.shape)
print("New shape:", val_dP_outputs.shape)

In [ ]:
# Add number of features (1) to outputs (targets)
val_dP_inputs = val_dP_inputs.permute(0, 3, 1, 2) # Shape will be [500, 1, 96, 200]
val_dP_outputs = val_dP_outputs[:, :, :].unsqueeze(1) # Shape will be [500, 1, 96, 200]

# Print new shape
print("New shape:", val_dP_inputs.shape)   # Expected: [num_samples, num_features, height, width] -> All conditions
print("New shape:", val_dP_outputs.shape)  # Expected: [num_samples, num_features, height, width] -> Only Pressure Build-Up @ year 30.0

In [ ]:
# Apply dnorm_dP function to ensure proper Pressure Build-Up range
val_dP_outputs=dnorm_dP(val_dP_outputs)

# Check Min and Max values 
min_val_dP = val_dP_outputs.min()
max_val_dP = val_dP_outputs.max()
print(min_val_dP) 
print(max_val_dP)

In [ ]:
# Some dP values are negative. Replace negative values with zeros
val_dP_outputs = torch.clamp(val_dP_outputs, min=0)

# Check Min and Max values 
min_val_dP = val_dP_outputs.min()
max_val_dP = val_dP_outputs.max()
print(min_val_dP) 
print(max_val_dP)

In [ ]:
# Define file path to save the min-max values BEFORE local normalization for the conditions 
save_path = os.path.join(data_dir_dP_val, "dP_val_conditions_min_max_before_normalization.txt")

# Open the file for writing
with open(save_path, "w") as f:
    f.write("Condition Min-Max Values Per Sample\n")
    f.write("=" * 50 + "\n")

    num_samples = val_dP_inputs.shape[0]  # Number of samples
    num_conditions = min(val_dP_inputs.shape[-1], 12) # Number of conditions (12)

    for i in range(num_conditions):  
        f.write(f"\nCondition {i+1}:\n")  
        f.write("-" * 50 + "\n")

        # Extract condition `i`
        condition_i = val_dP_inputs[:, i, :, :]  # Shape: [500, 96, 200]

        # Compute min and max values for each sample
        min_vals = condition_i.amin(dim=(1, 2))  # Min per sample
        max_vals = condition_i.amax(dim=(1, 2))  # Max per sample

        # Write per-sample min and max values
        for sample_idx in range(num_samples):
            min_val = min_vals[sample_idx].item()
            max_val = max_vals[sample_idx].item()

            # Write to the text file
            f.write(f"Sample {sample_idx+1}: Min: {min_val}, Max: {max_val}\n")

print(f"Condition min-max values saved in: {save_path}")

In [ ]:
# Define file path to save the min-max values for porosity BEFORE local normalization
save_path = os.path.join(data_dir_dP_val, "dP_val_porosity_min_max_before_normalization.txt")

# Open the file for writing
with open(save_path, "w") as f:
    f.write("Porosity Min-Max Values Per Sample\n")
    f.write("=" * 50 + "\n")

    num_samples = val_dP_inputs.shape[0]  # Number of samples

    # Initialize lists to store min/max values for porosity (Condition Index 2)
    min_val_por = []
    max_val_por = []

    # Extract porosity condition (Index 2)
    porosity_condition = val_dP_inputs[:, 2, :, :]  # Shape: [4500, 96, 200]

    # Compute per-sample min and max values for porosity
    for i in range(num_samples):
        min_val = porosity_condition[i].amin().item()  # Min for sample i
        max_val = porosity_condition[i].amax().item()  # Max for sample i
        min_val_por.append(min_val)
        max_val_por.append(max_val)

        # Write only porosity min/max values to the file
        f.write(f"Sample {i+1}: Min Porosity = {min_val}, Max Porosity = {max_val}\n")

print(f"Porosity min-max values saved in: {save_path}")

In [ ]:
# Define file path to save the min-max values AFTER local normalization
save_path = os.path.join(data_dir_dP_val, "dP_val_conditions_min_max_after_normalization.txt")

# Get shapes
num_samples = val_dP_inputs.shape[0]  # 4500
num_conditions = val_dP_inputs.shape[1]  # 12

# Local normalization: Normalize each sample's condition between -1 and 1
for i in range(num_conditions):
    condition_i = val_dP_inputs[:, i, :, :]  # Shape: [4500, 96, 200]

    # Compute per-sample min and max values
    min_vals = condition_i.amin(dim=(1, 2), keepdim=True)  # Shape: [4500, 1, 1]
    max_vals = condition_i.amax(dim=(1, 2), keepdim=True)  # Shape: [4500, 1, 1]

    # Compute denominator (range)
    diff = max_vals - min_vals

    # Fix: Handle constant conditions where min == max
    mask = diff == 0  # Identifies samples where max == min
    diff = diff.masked_fill(mask, 1)  # Prevent zero division

    # Normalize per sample
    normalized_condition = 2 * (condition_i - min_vals) / diff - 1

    # Keep original values when constant 
    normalized_condition.masked_scatter_(mask.expand_as(normalized_condition), condition_i)

    # Store back
    val_dP_inputs[:, i, :, :] = normalized_condition

# Save min-max values AFTER local normalization
with open(save_path, "w") as f:
    f.write("Condition Min-Max Values AFTER Local Normalization (Per Sample)\n")
    f.write("=" * 50 + "\n")

    for i in range(num_conditions):  
        f.write(f"\nCondition {i+1} (After Local Normalization):\n")  
        f.write("-" * 50 + "\n")

        condition_i = val_dP_inputs[:, i, :, :]
        min_vals = condition_i.amin(dim=(1, 2))
        max_vals = condition_i.amax(dim=(1, 2))

        for sample_idx in range(num_samples):
            min_val = min_vals[sample_idx].item()
            max_val = max_vals[sample_idx].item()
            f.write(f"Sample {sample_idx+1}: Min: {min_val}, Max: {max_val}\n")

print(f"Condition min-max values after local normalization saved in: {save_path}")

In [ ]:
# Check min and max values after global normalization
# To understand if the variable is a 2D map or a scalar you need to check the min and max values
# If min and max values are constant then the variable is a scalar 
# If the min and max values are different then the variable is a 2D map 
print("\nChecking global min/max for each condition after normalization:")

for i in range(num_conditions):
    global_min = val_dP_inputs[:, i, :, :].amin().item()
    global_max = val_dP_inputs[:, i, :, :].amax().item()
    print(f"Condition {i+1}: Global Min = {global_min}, Global Max = {global_max}")

In [ ]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_val_dP_individual = val_dP_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_val_dP_individual = val_dP_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_val_dP_individual: {min_val_dP_individual.shape}")  # [500]
print(f"Shape of max_val_dP_individual: {max_val_dP_individual.shape}")  # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_val_dP_individual, max_val_dP_individual)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in val_dP_outputs before local normalization
print("Global Min for Pressure Build-Up before normalization:", val_dP_outputs.min().item(), "Global Max for Pressure Build-Up before normalization:", val_dP_outputs.max().item())

In [ ]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
dP_log_file = os.path.join(log_dir, "val_dP_min_max_before_normalization.txt")

# Save min and max values before normalization into a text file
with open(dP_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values Before Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_val_dP_individual)):
        log_file.write(f"Sample {i+1}: Min Before={min_val_dP_individual[i].item():.4f}, Max Before={max_val_dP_individual[i].item():.4f}\n")

In [ ]:
# Normalize outputs between -1 and 1 

# Initialize arrays to store min and max values for each sample
min_val_dP = []
max_val_dP = []

# Normalize outputs and store min/max values
for i in range(len(val_dP_outputs)):
    min_val = val_dP_outputs[i].min()
    max_val = val_dP_outputs[i].max()
    print(f"Sample {i + 1}:")
    print(f"Original Min: {min_val}, Original Max: {max_val}")
    min_val_dP.append(min_val)
    max_val_dP.append(max_val)

    # Local Normalization formula
    val_dP_outputs[i] = 2 * (val_dP_outputs[i] - min_val) / (max_val - min_val) - 1
    print(f"Normalized Output: {val_dP_outputs[i]}")

In [ ]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_val_dP_individual_after_norm = val_dP_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_val_dP_individual_after_norm = val_dP_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_val_dP_individual_after_norm: {min_val_dP_individual_after_norm.shape}")  # [500]
print(f"Shape of max_val_dP_individual_after_norm: {max_val_dP_individual_after_norm.shape}")  # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_val_dP_individual_after_norm, max_val_dP_individual_after_norm)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in val_dP_outputs after local normalization
print("Global Min for Pressure Build-Up after normalization:", val_dP_outputs.min().item(), "Global Min for Pressure Build-Up after normalization:", val_dP_outputs.max().item())

In [ ]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
dP_log_file = os.path.join(log_dir, "val_dP_min_max_after_normalization.txt")

# Save min and max values before normalization into a text file
with open(dP_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values After Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_val_dP_individual_after_norm)):
        log_file.write(f"Sample {i+1}: Min After={min_val_dP_individual_after_norm[i].item():.4f}, Max After={max_val_dP_individual_after_norm[i].item():.4f}\n")

In [ ]:
# Define variables for conditional diffusion model
val_dP_label = val_dP_inputs
val_dP_target = val_dP_outputs

### 🗂️ Define Validation Dataset and Dataloader

In [ ]:
# Create dataloaders 
validation_dataset = TensorDataset(val_dP_target, val_dP_label)

### 🚀 Load Pre-Trained VQ-VAE + GAN Model 

In [ ]:
# Load pre-trained VQVAE checkpoint
vqvae_checkpoint = '/.../trained_models/pressure_buildup_models/vqvae/best_model/vqvae_model.pth'
vqvae_checkpoint = torch.load(vqvae_checkpoint)
# Extract parameters and weights
z_channels = vqvae_checkpoint["z_channels"]
codebook_size = vqvae_checkpoint["codebook_size"]
vqvae_state_dict = vqvae_checkpoint["state_dict"]
# Initialize the model with the saved parameters
vqvae_model = VQVAE(z_channels=z_channels, codebook_size=codebook_size)
# Load the model weights
vqvae_model.load_state_dict(vqvae_state_dict)
# Print successful loading
print("Model loaded successfully with z_channels={} and codebook_size={}".format(z_channels, codebook_size))

### 🚀 Load Pre-Trained Diffusion Model

In [ ]:
# Load pre-trained Unet model checkpoint
model_unet_checkpoint = "/.../trained_models/pressure_buildup_models/diffusion_model/best_model/diffusion_model.pth"
model_unet_checkpoint = torch.load(model_unet_checkpoint)

# Define parameters 
T = 1000 # Time steps
beta_start=0.0001
beta_end=0.02

# Extract parameters and weights
ddpm_checkpoint = model_unet_checkpoint["ddpm"]

# If `ddpm_checkpoint` is already an instance of a scheduler, just use it
if isinstance(ddpm_checkpoint, (Linear_Variance_Scheduler, Cosine_Variance_Scheduler)):
    ddpm = ddpm_checkpoint
else:
    # If `ddpm_checkpoint` is a string, initialize a new scheduler
    if ddpm_checkpoint == "linear":
        ddpm = Linear_Variance_Scheduler(time_steps=T, beta_start=beta_start, beta_end=beta_end)
    elif ddpm_checkpoint == "cosine":
        ddpm = Cosine_Variance_Scheduler(time_steps=T, s=0.008)
    else:
        raise ValueError(f"Unknown ddpm type: {ddpm_checkpoint}")

model_unet_state_dict = model_unet_checkpoint["state_dict"]

# Initialize diffusion model
model_unet = Conditional_Unet(t_emb_dim=256, z_channels=z_channels, image_cond=True, scalar_cond=False)

# Load the model weights
model_unet.load_state_dict(model_unet_state_dict)

# Print successful loading
print("Model loaded successfully with ddpm={}".format(ddpm))

### 🔗 Save Reverse Generation Samples

In [ ]:
# Define paths and parameters
save_dir = "/.../results/diffusion_model_pressure_buildup_results/reverse_generation_validation_dataset"  # Directory to save plots

# Specify the first 500 sample indices
sample_indices = list(range(500))

# Define reverse diffusion timesteps
steps = [1000, 900, 800, 700, 600, 500, 400, 300, 200, 100, 1]

In [ ]:
# Call the function
visualize_reverse_generation_dP_val(
    model_unet=model_unet,
    vqvae_model=vqvae_model,
    ddpm=ddpm,
    dataset=validation_dataset,
    min_val_dP=min_val_dP,
    max_val_dP=max_val_dP,
    min_val_por=min_val_por,
    max_val_por=max_val_por,
    save_dir=save_dir,
    sample_indices=sample_indices,
    steps=steps
)

## References 

**[1] Wen et al., 2002** ([Paper](https://www.sciencedirect.com/science/article/pii/S0309170822000562)): CO₂ Open-Source Dataset ([Google Drive Link](https://drive.google.com/drive/folders/1fZQfMn_vsjKUXAfRV0q_gswtl8JEkVGo)).

[2] How to train a Latent Conditional Diffusion Model ([GitHub Link](https://github.com/explainingai-code/StableDiffusion-PyTorch/tree/main)).